In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw dataset
df = pd.read_csv('../data/raw/hospital_readmissions.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (18330, 12)

Columns: ['Facility Name', 'Facility ID', 'State', 'Measure Name', 'Number of Discharges', 'Footnote', 'Excess Readmission Ratio', 'Predicted Readmission Rate', 'Expected Readmission Rate', 'Number of Readmissions', 'Start Date', 'End Date']


,Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137.0,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273.0,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024
3,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-COPD-HRRP,122.0,NaN,0.9823,16.6384,16.9389,19,07/01/2021,06/30/2024
4,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-PN-HRRP,507.0,NaN,0.9871,15.7529,15.9591,79,07/01/2021,06/30/2024


In [2]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nUnique conditions:", df['Measure Name'].unique())

Shape: (18330, 12)

Missing values:
 Facility Name                     0
Facility ID                       0
State                             0
Measure Name                      0
Number of Discharges          10088
Footnote                      11343
Excess Readmission Ratio       6610
Predicted Readmission Rate     6610
Expected Readmission Rate      6610
Number of Readmissions         6610
Start Date                        0
End Date                          0
dtype: int64

Data types:
 Facility Name                  object
Facility ID                     int64
State                          object
Measure Name                   object
Number of Discharges          float64
Footnote                      float64
Excess Readmission Ratio      float64
Predicted Readmission Rate    float64
Expected Readmission Rate     float64
Number of Readmissions         object
Start Date                     object
End Date                       object
dtype: object

Unique conditions: ['READM-30-HIP

In [3]:
# ── 1. Drop Footnote column — not useful ──────────────────────────────────
df = df.drop(columns=['Footnote'])

# ── 2. Drop rows where core metrics are missing (too few to report) ────────
df = df.dropna(subset=[
    'Excess Readmission Ratio',
    'Predicted Readmission Rate',
    'Expected Readmission Rate',
    'Number of Readmissions'
])

# ── 3. Clean column names — lowercase, underscores, no spaces ─────────────
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_'))

# ── 4. Convert dates to datetime ───────────────────────────────────────────
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date'] = pd.to_datetime(df['end_date'])

# ── 5. Clean up condition names — readable labels ─────────────────────────
condition_map = {
    'READM-30-HIP-KNEE-HRRP': 'Hip/Knee Replacement',
    'READM-30-CABG-HRRP':     'Heart Bypass Surgery',
    'READM-30-AMI-HRRP':      'Heart Attack',
    'READM-30-COPD-HRRP':     'COPD',
    'READM-30-PN-HRRP':       'Pneumonia',
    'READM-30-HF-HRRP':       'Heart Failure'
}
df['measure_name'] = df['measure_name'].map(condition_map)

# ── 6. Add risk category based on Excess Readmission Ratio ────────────────
def risk_category(ratio):
    if ratio > 1.10:
        return 'High Risk'
    elif ratio > 1.00:
        return 'Above Average'
    elif ratio > 0.90:
        return 'Below Average'
    else:
        return 'Low Risk'

df['risk_category'] = df['excess_readmission_ratio'].apply(risk_category)

# ── 7. Verify ──────────────────────────────────────────────────────────────
print("Clean shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nRisk categories:\n", df['risk_category'].value_counts())
df.head()

Clean shape: (11720, 12)

Missing values:
 facility_name                    0
facility_id                      0
state                            0
measure_name                     0
number_of_discharges          3683
excess_readmission_ratio         0
predicted_readmission_rate       0
expected_readmission_rate        0
number_of_readmissions           0
start_date                       0
end_date                         0
risk_category                    0
dtype: int64

Risk categories:
 risk_category
Below Average    5201
Above Average    4627
High Risk        1016
Low Risk          876
Name: count, dtype: int64


,facility_name,facility_id,state,measure_name,number_of_discharges,excess_readmission_ratio,predicted_readmission_rate,expected_readmission_rate,number_of_readmissions,start_date,end_date,risk_category
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Hip/Knee Replacement,NaN,0.9875,4.5734,4.6311,Too Few to Report,2021-07-01,2024-06-30,Below Average
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Heart Bypass Surgery,137.0,0.9531,10.3960,10.9078,13,2021-07-01,2024-06-30,Below Average
2,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Heart Attack,273.0,0.9370,13.2998,14.1948,33,2021-07-01,2024-06-30,Below Average
3,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,COPD,122.0,0.9823,16.6384,16.9389,19,2021-07-01,2024-06-30,Below Average
4,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Pneumonia,507.0,0.9871,15.7529,15.9591,79,2021-07-01,2024-06-30,Below Average


In [4]:
# ── Fix number_of_readmissions — replace string with NaN, convert to numeric
df['number_of_readmissions'] = pd.to_numeric(
    df['number_of_readmissions'], errors='coerce'
)

# ── Fix number_of_discharges — same treatment
df['number_of_discharges'] = pd.to_numeric(
    df['number_of_discharges'], errors='coerce'
)

# ── Now drop rows where readmissions is still NaN ─────────────────────────
df = df.dropna(subset=['number_of_readmissions'])

# ── For discharges, fill remaining NaN with median per condition ───────────
df['number_of_discharges'] = df.groupby('measure_name')['number_of_discharges']\
    .transform(lambda x: x.fillna(x.median()))

# ── Verify ─────────────────────────────────────────────────────────────────
print("Final shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nSample readmissions:", df['number_of_readmissions'].dtype)
print(df['number_of_readmissions'].describe())

Final shape: (8037, 12)

Missing values:
 facility_name                 0
facility_id                   0
state                         0
measure_name                  0
number_of_discharges          0
excess_readmission_ratio      0
predicted_readmission_rate    0
expected_readmission_rate     0
number_of_readmissions        0
start_date                    0
end_date                      0
risk_category                 0
dtype: int64

Sample readmissions: float64
count    8037.000000
mean       48.750653
std        49.758206
min        11.000000
25%        19.000000
50%        31.000000
75%        60.000000
max       847.000000
Name: number_of_readmissions, dtype: float64


In [6]:
# Export clean dataset
df.to_csv('../data/clean/hospital_readmissions_clean.csv', index=False)

print("File saved successfully")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nCondition breakdown:")
print(df['measure_name'].value_counts())
print("\nRisk breakdown:")
print(df['risk_category'].value_counts())

File saved successfully
Final dataset: 8037 rows x 12 columns

Condition breakdown:
measure_name
Pneumonia               2320
Heart Failure           2316
COPD                    1545
Heart Attack            1240
Heart Bypass Surgery     363
Hip/Knee Replacement     253
Name: count, dtype: int64

Risk breakdown:
risk_category
Above Average    3567
Below Average    3255
High Risk         748
Low Risk          467
Name: count, dtype: int64
